# Credit Events and Restructuring

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Purpose:** Show three common distressed-credit workflows with a Rust-owned recovery waterfall and notebook-level decision policy.

**Prerequisites:** Basic familiarity with capital-structure priority, exchange offers, and liability management exercises.

**In this notebook:** We run the canonical recovery waterfall, compare hold-versus-tender economics for an exchange offer, and quantify an open-market repurchase LME.

> These are decision-support calculations, not mark-to-market valuations. Recovery allocation and validation live in Rust; recommendation policy remains explicit in the notebook.


## Concept

These analyses answer three common distressed-credit questions:

1. **Recovery waterfall** — who recovers what under a stressed distributable value, following the Absolute Priority Rule?
2. **Exchange offer** — is a hold-vs-tender trade-off economically attractive to holders?
3. **LME** — how much debt reduction and discount capture does a discounted liability-management move create?

In [1]:
## Helpers

from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional


def banner(title: str) -> None:
    print(f"\n{'=' * 72}")
    print(title)
    print("=" * 72)


def fmt_money(value: float) -> str:
    return f"${value:,.0f}"


def fmt_pct(value: float) -> str:
    return f"{value * 100:.2f}%"

## Recovery waterfall

The executable setup below imports the Rust-backed recovery API and maps the notebook's seniority labels to explicit numeric priorities. Lower numeric priorities recover first; equal-priority deficiencies share an insufficient residual estate pro rata.

The distributable estate includes pledged collateral. Haircut net collateral is allocated first and deducted from the estate before general recovery, preventing double counting.

In [2]:
## Exchange Offer and LME Helpers


def analyze_exchange_offer(
    old_pv: float,
    new_pv: float,
    consent_fee: float = 0.0,
    equity_sweetener_value: float = 0.0,
    exchange_type: str = "par_for_par",
) -> dict:
    """Compare hold-vs-tender economics for a distressed exchange offer.

    old_pv: present value of the existing claim if not tendered.
    new_pv: present value of the new instrument received.
    consent_fee: cash consent / early-tender fee.
    equity_sweetener_value: estimated value of attached equity/warrants.
    exchange_type: one of par_for_par, discount, uptier, downtier.
    """
    valid_types = {"par_for_par", "discount", "uptier", "downtier"}
    canonical = exchange_type.lower().replace("-", "_")
    if canonical == "par":
        canonical = "par_for_par"
    if canonical not in valid_types:
        raise ValueError(f"Unknown exchange_type '{exchange_type}' — expected one of {valid_types}")
    if any(v < 0 for v in [old_pv, new_pv, consent_fee, equity_sweetener_value]):
        raise ValueError("old_pv, new_pv, consent_fee, equity_sweetener_value must be non-negative")

    tender_total = new_pv + consent_fee + equity_sweetener_value
    delta_npv = tender_total - old_pv
    breakeven_recovery = min(tender_total / old_pv, 1.0) if old_pv > 0 else 1.0
    tender_recommended = tender_total > old_pv * 1.02

    return {
        "exchange_type": canonical,
        "old_npv": old_pv,
        "new_npv": new_pv,
        "consent_fee": consent_fee,
        "equity_sweetener_value": equity_sweetener_value,
        "tender_total": tender_total,
        "delta_npv": delta_npv,
        "breakeven_recovery": breakeven_recovery,
        "tender_recommended": tender_recommended,
    }


def analyze_lme(
    lme_type: str,
    notional: float,
    repurchase_price_pct: float,
    opt_acceptance_pct: float = 1.0,
    ebitda: Optional[float] = None,
) -> dict:
    """Compute discount capture and leverage impact for an LME transaction.

    lme_type: open_market, tender_offer, amend_and_extend, or dropdown.
    notional: outstanding notional of the target instrument.
    repurchase_price_pct: price as fraction of par (0.60 = 60 cents).
    opt_acceptance_pct: fraction of holders participating (0.0–1.0).
    ebitda: if provided, leverage_impact block is included in the output.
    """
    if notional <= 0:
        raise ValueError(f"notional must be positive, got {notional}")
    if not 0.0 <= opt_acceptance_pct <= 1.0:
        raise ValueError(f"opt_acceptance_pct must be in [0.0, 1.0], got {opt_acceptance_pct}")

    kind = lme_type.lower().replace("-", "_").replace("&", "_")
    participating = notional * opt_acceptance_pct

    if kind in ("open_market", "open_market_repurchase", "omr"):
        if not 0.0 < repurchase_price_pct <= 1.5:
            raise ValueError(f"repurchase_price_pct must be in (0.0, 1.5], got {repurchase_price_pct}")
        par_retired = participating
        cost = par_retired * repurchase_price_pct
        canonical = "open_market_repurchase"
        remaining_holder_pct = 0.0
    elif kind in ("tender_offer", "tender"):
        if not 0.0 < repurchase_price_pct <= 1.5:
            raise ValueError(f"repurchase_price_pct must be in (0.0, 1.5], got {repurchase_price_pct}")
        par_retired = participating
        cost = par_retired * repurchase_price_pct
        canonical = "tender_offer"
        remaining_holder_pct = 0.0
    elif kind in ("amend_and_extend", "ae", "a_e"):
        if not 0.0 <= repurchase_price_pct <= 0.10:
            raise ValueError(f"extension_fee must be in [0.0, 0.10], got {repurchase_price_pct}")
        par_retired = 0.0
        cost = participating * repurchase_price_pct
        canonical = "amend_and_extend"
        remaining_holder_pct = 0.0
    elif kind == "dropdown":
        if not 0.0 <= repurchase_price_pct <= 1.0:
            raise ValueError(f"transferred-asset fraction must be in [0.0, 1.0], got {repurchase_price_pct}")
        par_retired = 0.0
        cost = 0.0
        canonical = "dropdown"
        remaining_holder_pct = repurchase_price_pct
    else:
        raise ValueError(
            f"Unknown lme_type '{lme_type}' — expected open_market, tender_offer, amend_and_extend, dropdown"
        )

    discount_capture = par_retired - cost
    discount_capture_pct = discount_capture / par_retired if par_retired > 0 else 0.0

    result: dict = {
        "lme_type": canonical,
        "cost": cost,
        "notional_reduction": par_retired,
        "discount_capture": discount_capture,
        "discount_capture_pct": discount_capture_pct,
        "remaining_holder_impact_pct": remaining_holder_pct,
        "leverage_impact": None,
    }

    if ebitda and ebitda > 0:
        post_debt = notional - par_retired
        result["leverage_impact"] = {
            "pre_total_debt": notional,
            "post_total_debt": post_debt,
            "pre_leverage": notional / ebitda,
            "post_leverage": post_debt / ebitda,
            "leverage_reduction": (notional - post_debt) / ebitda,
        }

    return result

## Takeaways

- **`allocate_recovery()`** applies collateral and distributes the remaining estate by absolute priority in the canonical Rust kernel.
- **`analyze_exchange_offer()`** reframes a restructuring proposal in holder economics rather than issuer rhetoric.
- **`analyze_lme()`** measures debt reduction and leverage change from discounted liability-management moves.

The finance and validation kernel is reusable and tested in Rust. Holder recommendation policy remains visible and editable in this notebook.

In [3]:
import json
from pathlib import Path

from finstack_quant.core.credit.recovery_waterfall import (
    RecoveryClaim,
    allocate_recovery,
)

_NOTEBOOK_DATA = json.loads(Path("data/credit_events_and_restructuring.json").read_text())
SENIORITY_PRIORITY = {seniority: priority for priority, seniority in enumerate(_NOTEBOOK_DATA["SENIORITY_ORDER"])}

claims = [
    RecoveryClaim(
        id=claim.get("id", f"claim-{index}"),
        seniority=claim["seniority"],
        priority=SENIORITY_PRIORITY[claim["seniority"]],
        principal=claim["principal"],
        accrued=claim.get("accrued", 0.0),
        penalties=claim.get("penalties", 0.0),
        collateral=(claim["collateral_value"], claim.get("haircut", 0.0)) if "collateral_value" in claim else None,
    )
    for index, claim in enumerate(_NOTEBOOK_DATA["claims"])
]

In [4]:
## Example Run

banner("Recovery waterfall")
waterfall = allocate_recovery(100_000_000.0, claims)
print(f"Total distributed: {fmt_money(waterfall.total_distributed)}")
print(f"Residual         : {fmt_money(waterfall.undistributed_estate)}")
for row in waterfall.allocations:
    print(
        f"  {row.id:<20} claim={fmt_money(row.total_claim)} "
        f"recovery={fmt_money(row.total_recovery)} rate={fmt_pct(row.recovery_rate)}"
    )

banner("Exchange offer")
exchange = analyze_exchange_offer(
    old_pv=45.0,
    new_pv=80.0,
    consent_fee=2.0,
    equity_sweetener_value=0.0,
    exchange_type="discount",
)
print(f"Tender total     : ${exchange['tender_total']:.2f}")
print(f"Delta NPV        : ${exchange['delta_npv']:+.2f}")
print(f"Breakeven recov. : {fmt_pct(exchange['breakeven_recovery'])}")
print(f"Tender?          : {exchange['tender_recommended']}")

banner("Open-market repurchase LME")
lme = analyze_lme(
    lme_type="open_market",
    notional=200_000_000.0,
    repurchase_price_pct=0.60,
    opt_acceptance_pct=0.40,
    ebitda=25_000_000.0,
)
print(f"Cash cost        : {fmt_money(lme['cost'])}")
print(f"Notional retired : {fmt_money(lme['notional_reduction'])}")
print(f"Discount capture : {fmt_money(lme['discount_capture'])}")
print(f"Holder impact    : {fmt_pct(lme['remaining_holder_impact_pct'])}")
if lme["leverage_impact"] is not None:
    leverage = lme["leverage_impact"]
    print(f"Pre leverage     : {leverage['pre_leverage']:.2f}x")
    print(f"Post leverage    : {leverage['post_leverage']:.2f}x")

# Summary for test runner
{
    "waterfall_apr_satisfied": waterfall.apr_satisfied,
    "exchange_delta_npv": round(exchange["delta_npv"], 2),
    "lme_discount_capture": round(lme["discount_capture"], 2),
}


Recovery waterfall
Total distributed: $100,000,000
Residual         : $0
  first_lien_tl        claim=$51,000,000 recovery=$51,000,000 rate=100.00%
  sr_unsec_notes       claim=$82,000,000 recovery=$49,000,000 rate=59.76%
  sub_notes            claim=$41,000,000 recovery=$0 rate=0.00%
  common_equity        claim=$0 recovery=$0 rate=0.00%

Exchange offer
Tender total     : $82.00
Delta NPV        : $+37.00
Breakeven recov. : 100.00%
Tender?          : True

Open-market repurchase LME
Cash cost        : $48,000,000
Notional retired : $80,000,000
Discount capture : $32,000,000
Holder impact    : 0.00%
Pre leverage     : 8.00x
Post leverage    : 4.80x


{'waterfall_apr_satisfied': True,
 'exchange_delta_npv': 37.0,
 'lme_discount_capture': 32000000.0}